# Ice-Thickness Sweep at 10 Hz

This notebook compares the signed reflection-coefficient estimate across the `10 Hz` ice-thickness sweep. The floating-side cavity water remains `100 m` thick, the free surface stays fixed, and only the ice thickness changes.

The requested thickness set is:
`75, 100, 125, 150, 175, 250, 325, 400 m`

The notebook is written to handle partially completed sweeps gracefully: it will plot the thicknesses that already exist in `summary.csv`, list any pending thicknesses, and only attempt trace/gather plots for completed cases.

In [ ]:
from csv import DictReader
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
summary_path = ROOT / 'results_icethickness_10hz' / 'summary.csv'
reference_path = ROOT / 'results_supportedcavity_10hz' / 'summary.csv'
requested_thicknesses = np.asarray([75, 100, 125, 150, 175, 250, 325, 400], dtype=float)

def load_rows(path):
    with path.open() as handle:
        return list(DictReader(handle))

rows = sorted(load_rows(summary_path), key=lambda row: float(row['ice_thickness_m']))
reference_rows = load_rows(reference_path)
completed_thicknesses = np.asarray([float(row['ice_thickness_m']) for row in rows])
pending_thicknesses = [h for h in requested_thicknesses if h not in set(completed_thicknesses)]

print(f'Loaded {len(rows)} completed thickness cases')
print(f'Loaded {len(reference_rows)} reference material-sweep cases at 10 Hz')
print('Completed thicknesses:', completed_thicknesses.tolist())
print('Pending thicknesses:', pending_thicknesses if pending_thicknesses else 'none')
rows

In [ ]:
ice_thickness = np.asarray([float(row['ice_thickness_m']) for row in rows])
reflection = np.asarray([float(row['reflection_coefficient']) for row in rows])
legacy_reflection = np.asarray([
    float(row.get('legacy_reflection_coefficient', row['reflection_coefficient'])) for row in rows
])
correlation = np.asarray([float(row['correlation']) for row in rows])
alignment_lag = np.asarray([float(row.get('alignment_lag_s', 0.0)) for row in rows])
ice_base = np.asarray([float(row['ice_base_z']) for row in rows])
material_reflection = np.asarray([float(row['reflection_coefficient']) for row in reference_rows])

for row in rows:
    print(
        f"{row['case_id']:>8s}  H={float(row['ice_thickness_m']):6.1f} m  "
        f"R_new={float(row['reflection_coefficient']): .6f}  "
        f"R_old={float(row.get('legacy_reflection_coefficient', row['reflection_coefficient'])): .6f}  "
        f"corr={float(row['correlation']): .3f}  "
        f"lag={float(row.get('alignment_lag_s', 0.0)): .4f} s"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ice_thickness, legacy_reflection, 'o--', lw=1.5, ms=7, color='0.5', label='legacy phase-sensitive estimate')
ax.plot(ice_thickness, reflection, 'o-', lw=2, ms=8, color='tab:blue', label='current envelope-aligned estimate')
ax.fill_between(
    [requested_thicknesses.min(), requested_thicknesses.max()],
    material_reflection.min(),
    material_reflection.max(),
    color='tab:orange',
    alpha=0.20,
    label='10 Hz material-sweep range (supported cavity)',
)
ax.axhline(material_reflection.mean(), color='tab:orange', ls='--', lw=1.5, label='material-sweep mean')
if pending_thicknesses:
    ax.scatter(pending_thicknesses, np.full(len(pending_thicknesses), material_reflection.mean()),
               marker='x', s=80, color='gray', label='requested but not yet completed')
ax.axhline(0.0, color='k', lw=1.0, alpha=0.5)
ax.set_xlabel('ice thickness (m)')
ax.set_ylabel('estimated reflection coefficient')
ax.set_title('Reflection coefficient versus ice thickness at 10 Hz')
ax.set_xlim(requested_thicknesses.min() - 5, requested_thicknesses.max() + 10)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, constrained_layout=True)
ax1.plot(ice_thickness, legacy_reflection, 'o--', color='0.5', lw=1.5, label='legacy estimate')
ax1.plot(ice_thickness, reflection, 'o-', color='tab:blue', lw=2, label='current estimate')
ax1.axhline(0.0, color='k', lw=1.0, alpha=0.5)
ax1.set_ylabel('reflection coefficient')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='best')

ax2.plot(ice_thickness, correlation, 's-', color='tab:red', lw=1.5, label='aligned waveform correlation')
ax2.axhline(0.0, color='k', lw=1.0, alpha=0.3)
ax2.set_xlabel('ice thickness (m)')
ax2.set_ylabel('correlation', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.set_xlim(requested_thicknesses.min() - 5, requested_thicknesses.max() + 10)
ax2.grid(True, alpha=0.3)

ax3 = ax2.twinx()
ax3.plot(ice_thickness, alignment_lag, 'd--', color='tab:green', lw=1.2, label='best alignment lag')
ax3.set_ylabel('alignment lag (s)', color='tab:green')
ax3.tick_params(axis='y', labelcolor='tab:green')

fig.suptitle('Diagnostic view of the corrected reflection estimator')

In [ ]:
def load_gather(case_id):
    path = ROOT / 'results_icethickness_10hz' / case_id / 'surface_gather.npz'
    data = np.load(path, allow_pickle=True)
    return {key: data[key] for key in data.files}

def trace_at_x(gather, x_target):
    idx = int(np.argmin(np.abs(gather['x'] - x_target)))
    return gather['x'][idx], gather['bxx'][idx, :]

selected_indices = np.unique(np.linspace(0, len(rows) - 1, min(3, len(rows)), dtype=int))
selected_case_ids = [rows[i]['case_id'] for i in selected_indices]
selected_x = -1000.0
fig, axes = plt.subplots(len(selected_case_ids), 1, figsize=(10, 3.2 * len(selected_case_ids)), sharex=False, constrained_layout=True)
for ax, case_id in zip(np.atleast_1d(axes), selected_case_ids):
    gather = load_gather(case_id)
    x_used, trace = trace_at_x(gather, selected_x)
    trace = trace / max(np.max(np.abs(trace)), 1e-12)
    h = next(float(row['ice_thickness_m']) for row in rows if row['case_id'] == case_id)
    ax.plot(gather['time'], trace, color='black', lw=1.0)
    ax.set_title(f'{case_id}: H = {h:.0f} m, trace near x = {x_used:.1f} m')
    ax.set_ylabel('normalized BXX')
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel('time (s)')

In [ ]:
selected_indices = np.unique(np.linspace(0, len(rows) - 1, min(3, len(rows)), dtype=int))
selected_case_ids = [rows[i]['case_id'] for i in selected_indices]
fig, axes = plt.subplots(1, len(selected_case_ids), figsize=(5 * len(selected_case_ids), 5), sharey=True, constrained_layout=True)
for ax, case_id in zip(np.atleast_1d(axes), selected_case_ids):
    gather = load_gather(case_id)
    clip = np.percentile(np.abs(gather['bxx']), 99)
    h = next(float(row['ice_thickness_m']) for row in rows if row['case_id'] == case_id)
    ax.imshow(
        gather['bxx'].T,
        origin='lower',
        aspect='auto',
        extent=[gather['x'].min(), gather['x'].max(), gather['time'].min(), gather['time'].max()],
        vmin=-clip, vmax=clip,
    )
    ax.set_title(f'H = {h:.0f} m')
    ax.set_xlabel('x (m)')
axes[0].set_ylabel('time (s)')
fig.suptitle('Representative 10 Hz surface gathers from the completed thickness cases')